# Deep Learning Capstone: RNN vs Transformer
**Task:** Empirically demonstrate RNN limitations → Solve with DistilBERT → Explain with numbers

**Dataset:** IMDB Sentiment (50k reviews, binary classification)

---

## 0. Environment Setup

In [1]:
!pip install transformers datasets tensorflow torch --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.1 which is incompatible.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

# TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# HuggingFace Transformers — requires transformers >= 4.30.0
from transformers import (
    DistilBertTokenizerFast,
    TFDistilBertForSequenceClassification,
    DistilBertModel
)

# PyTorch (used only for attention weight extraction)
import torch

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# ── Verify everything loaded correctly ──────────────────────────────────────
import transformers
print(f"Transformers : {transformers.__version__}")
print(f"TensorFlow   : {tf.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("\n✓ All imports successful")

ImportError: cannot import name 'TFDistilBertForSequenceClassification' from 'transformers' (c:\Users\mnour\anaconda3\Lib\site-packages\transformers\__init__.py)

---
## 1. Data Loading & Preprocessing

In [ ]:
DATA_PATH = r'C:\Users\mnour\Downloads\IMDB Dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Class distribution:\n{df["sentiment"].value_counts()}')
df.head(3)

In [ ]:
df['label'] = (df['sentiment'] == 'positive').astype(int)
df['review_len'] = df['review'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['review_len'], bins=80, color='steelblue', edgecolor='white')
axes[0].axvline(df['review_len'].median(), color='red', linestyle='--', label=f'Median: {df["review_len"].median():.0f}')
axes[0].set_title('Review Length Distribution'); axes[0].legend()
df['sentiment'].value_counts().plot(kind='bar', ax=axes[1], color=['steelblue','coral'], edgecolor='white')
axes[1].set_title('Class Balance'); axes[1].tick_params(rotation=0)
plt.tight_layout(); plt.show()
print(df['review_len'].describe().round(1))

In [ ]:
texts = df['review'].values
labels = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

---
## 2. Part 1 — LSTM Baseline
### 2a. Tokenization for RNN

In [ ]:
VOCAB_SIZE    = 20000
MAX_LEN_SHORT = 256
MAX_LEN_LONG  = 512
EMBED_DIM     = 128
LSTM_UNITS    = 64
BATCH_SIZE    = 64
LSTM_EPOCHS   = 5

tokenizer_rnn = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer_rnn.fit_on_texts(X_train)

def encode_for_rnn(texts, maxlen):
    seqs = tokenizer_rnn.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')

X_train_short = encode_for_rnn(X_train, MAX_LEN_SHORT)
X_test_short  = encode_for_rnn(X_test,  MAX_LEN_SHORT)
X_train_long  = encode_for_rnn(X_train, MAX_LEN_LONG)
X_test_long   = encode_for_rnn(X_test,  MAX_LEN_LONG)

print(f'Short sequences: {X_train_short.shape}')
print(f'Long  sequences: {X_train_long.shape}')

### 2b. LSTM Model Architecture

In [ ]:
def build_lstm(input_len, name='LSTM_Model'):
    """
    Stacked Bidirectional LSTM classifier.
    Embed -> BiLSTM -> BiLSTM -> GlobalMaxPool -> Dense -> Sigmoid
    """
    inp = keras.Input(shape=(input_len,))
    x = layers.Embedding(VOCAB_SIZE, EMBED_DIM)(inp)
    x = layers.SpatialDropout1D(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(LSTM_UNITS, return_sequences=True, dropout=0.2))(x)
    x = layers.Bidirectional(layers.LSTM(LSTM_UNITS // 2, return_sequences=True, dropout=0.2))(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out, name=name)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_lstm(MAX_LEN_SHORT, name='LSTM_Short')
lstm_model.summary()

### 2c. Train LSTM — Short Context (seq=256)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1)
]

t0 = time.time()
history_short = lstm_model.fit(
    X_train_short, y_train, validation_split=0.1,
    epochs=LSTM_EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=1
)
lstm_train_time_short = time.time() - t0
print(f'Training time (short): {lstm_train_time_short:.1f}s')

In [ ]:
t_inf = time.time()
y_pred_short = (lstm_model.predict(X_test_short, batch_size=256) > 0.5).astype(int).flatten()
lstm_inf_time_short = time.time() - t_inf

lstm_acc_short = accuracy_score(y_test, y_pred_short)
print(f'LSTM Accuracy (seq=256): {lstm_acc_short:.4f}')
print(classification_report(y_test, y_pred_short, target_names=['Negative','Positive']))

### 2d. LSTM Degradation — Long Context (seq=512)

In [ ]:
lstm_model_long = build_lstm(MAX_LEN_LONG, name='LSTM_Long')

t0 = time.time()
history_long = lstm_model_long.fit(
    X_train_long, y_train, validation_split=0.1,
    epochs=LSTM_EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=1
)
lstm_train_time_long = time.time() - t0

t_inf = time.time()
y_pred_long = (lstm_model_long.predict(X_test_long, batch_size=256) > 0.5).astype(int).flatten()
lstm_inf_time_long = time.time() - t_inf

lstm_acc_long = accuracy_score(y_test, y_pred_long)
print(f'LSTM Accuracy (seq=512): {lstm_acc_long:.4f}')
print(f'Accuracy drop due to longer context: {lstm_acc_short - lstm_acc_long:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, hist, title in zip(axes, [history_short, history_long],
                            ['LSTM (seq=256)', 'LSTM (seq=512)']):
    ax.plot(hist.history['accuracy'], label='Train', marker='o')
    ax.plot(hist.history['val_accuracy'], label='Val', marker='s', linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('LSTM Training Curves: Short vs Long Context', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3. Part 2 — Why Transformers Replaced RNNs

### Technical Reason 1: Parallelism

**RNNs are inherently sequential.** Hidden state $h_t$ depends on $h_{t-1}$:
$$h_t = f(W_h \cdot h_{t-1} + W_x \cdot x_t + b)$$
This creates $T$ sequential operations for sequence length $T$. Token 500 cannot be computed until tokens 1–499 finish — **GPU parallelism is wasted**.

**Transformers process all tokens simultaneously** via self-attention:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
All token pairs computed in one matrix multiplication: **O(1) sequential operations** vs RNN's O(T).

---

### Technical Reason 2: Long-Range Dependencies

**RNNs suffer vanishing gradients.** The gradient of loss w.r.t. $h_1$:
$$\frac{\partial L}{\partial h_1} = \prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$
For T=512, multiplying ~512 values ≤1 produces gradient ≈ **0**. LSTMs mitigate but don't eliminate this for T > 300.

**Transformers have direct O(1) connections** between any two tokens regardless of distance.

---

### Technical Reason 3: Gradient Flow via Residual Connections

**Every Transformer block uses residual connections + LayerNorm:**
$$\text{Output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$
Direct gradient highway through the network — enables training 96+ layer networks with stable convergence. LSTM gradient flow degrades with sequence length due to gate-dependent paths.

---
## 4. Part 3 — DistilBERT Fine-Tuning

In [ ]:
MODEL_NAME   = 'distilbert-base-uncased'
BERT_MAX_LEN = 256
BERT_BATCH   = 16
BERT_EPOCHS  = 3
SUBSET_SIZE  = 5000  # Set None for full dataset

if SUBSET_SIZE:
    idx = np.random.RandomState(42).choice(len(X_train), SUBSET_SIZE, replace=False)
    X_tr_bert, y_tr_bert = X_train[idx], y_train[idx]
    print(f'GPU-limited mode: {SUBSET_SIZE} training samples')
else:
    X_tr_bert, y_tr_bert = X_train, y_train
    print(f'Full training set: {len(X_train)} samples')

In [ ]:
bert_tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_batch(texts, tokenizer, max_len):
    return tokenizer(list(texts), max_length=max_len, padding='max_length',
                    truncation=True, return_tensors='tf')

print('Tokenizing training set...')
train_enc = tokenize_batch(X_tr_bert, bert_tokenizer, BERT_MAX_LEN)
print('Tokenizing test set...')
test_enc  = tokenize_batch(X_test, bert_tokenizer, BERT_MAX_LEN)
print(f'Train input_ids shape: {train_enc["input_ids"].shape}')

In [ ]:
def make_tf_dataset(encodings, labels, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((dict(encodings), labels))
    if shuffle: ds = ds.shuffle(1000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(train_enc, y_tr_bert, BERT_BATCH, shuffle=True)
test_ds  = make_tf_dataset(test_enc,  y_test,    BERT_BATCH, shuffle=False)
print(f'Train batches: {len(train_ds)} | Test batches: {len(test_ds)}')

In [ ]:
bert_model = TFDistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
bert_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
print(f'Total parameters: {bert_model.count_params():,}')

In [ ]:
t0 = time.time()
bert_history = bert_model.fit(
    train_ds, validation_data=test_ds, epochs=BERT_EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(patience=1, restore_best_weights=True)]
)
bert_train_time = time.time() - t0
print(f'DistilBERT training time: {bert_train_time:.1f}s')

In [ ]:
t_inf = time.time()
bert_logits  = bert_model.predict(test_ds).logits
bert_inf_time = time.time() - t_inf

y_pred_bert = np.argmax(bert_logits, axis=1)
bert_acc    = accuracy_score(y_test, y_pred_bert)

print(f'DistilBERT Accuracy: {bert_acc:.4f}')
print(f'Inference time: {bert_inf_time:.2f}s for {len(X_test)} samples')
print(classification_report(y_test, y_pred_bert, target_names=['Negative','Positive']))

---
## 5. Part 4 — Attention Visualization

In [ ]:
# Load PyTorch DistilBERT to access raw attention weights
pt_tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
pt_model     = DistilBertModel.from_pretrained(MODEL_NAME, output_attentions=True)
pt_model.eval()

sample_idx  = 0
sample_text = X_test[sample_idx]
label_map   = {0: 'Negative', 1: 'Positive'}
print(f'True: {label_map[y_test[sample_idx]]} | Predicted: {label_map[y_pred_bert[sample_idx]]}')
print(f'Review (first 200 chars): {sample_text[:200]}...')

In [ ]:
MAX_TOKENS_VIZ = 30

inputs = pt_tokenizer(sample_text, return_tensors='pt',
                      max_length=MAX_TOKENS_VIZ, truncation=True)
with torch.no_grad():
    outputs = pt_model(**inputs)

attentions = outputs.attentions  # 6 layers x [1, 12heads, seq, seq]
tokens     = pt_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f'Tokens ({len(tokens)}): {tokens}')

In [ ]:
# --- Plot 1: [CLS] attention to content tokens ---
last_attn    = attentions[-1][0]                    # [heads, seq, seq]
avg_attn     = last_attn.mean(dim=0).numpy()        # [seq, seq]
cls_attn     = avg_attn[0, 1:-1]                   # [CLS] -> content
content_toks = tokens[1:-1]

fig, ax = plt.subplots(figsize=(14, 4))
colors = plt.cm.YlOrRd(cls_attn / cls_attn.max())
ax.bar(range(len(content_toks)), cls_attn, color=colors, edgecolor='white')
ax.set_xticks(range(len(content_toks)))
ax.set_xticklabels(content_toks, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Attention Weight')
ax.set_title('[CLS] Token Attention — What the model focuses on for classification')
ax.grid(axis='y', alpha=0.3)

top3 = np.argsort(cls_attn)[-3:]
for i in top3:
    ax.text(i, cls_attn[i]+0.001, content_toks[i], ha='center',
            fontweight='bold', color='darkred', fontsize=8)
plt.tight_layout(); plt.show()

print('Top-5 attended tokens:')
for score, tok in sorted(zip(cls_attn, content_toks), reverse=True)[:5]:
    print(f"  '{tok}': {score:.4f}")

In [ ]:
# --- Plot 2: Full attention heatmap (last layer, head 0) ---
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(last_attn[0].numpy(), cmap='Blues', aspect='auto')
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(tokens, fontsize=8)
ax.set_title('Attention Heatmap — Layer 6, Head 0\nRow=Query Token | Col=Key Token', fontweight='bold')
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# --- Plot 3: All 6 heads (DistilBERT has 6 layers x 12 heads) ---
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for h, ax in enumerate(axes.flatten()):
    ax.imshow(last_attn[h].numpy(), cmap='Purples', aspect='auto')
    ax.set_title(f'Head {h+1}', fontweight='bold')
    ax.set_xticks(range(0, len(tokens), 3))
    ax.set_yticks(range(0, len(tokens), 3))
    ax.set_xticklabels(tokens[::3], rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(tokens[::3], fontsize=7)
plt.suptitle('All Attention Heads — Layer 6\nEach head captures different linguistic relationships',
             fontweight='bold')
plt.tight_layout(); plt.show()

### Attention Interpretation

**[CLS] attention chart:** Tokens with highest weight are the model's sentiment anchors. Sentiment-bearing words (e.g., 'excellent', 'terrible', 'not') dominate — confirming learned semantic focus.

**Head specialization patterns:**
- Diagonal → local context (adjacent token dependencies)
- Column focus → globally-attended anchor tokens
- [CLS]/[SEP] concentration → summary aggregation behavior

**Key advantage over LSTM:** Attention weights are **distance-independent**. The model can attend to any token at any position with equal fidelity — no vanishing distance effect.

---
## 6. Part 5 — Final Comparison Table

In [ ]:
lstm_params_short = lstm_model.count_params()
lstm_params_long  = lstm_model_long.count_params()
bert_params       = bert_model.count_params()
n_test = len(X_test)

lstm_ms_s = (lstm_inf_time_short / n_test) * 1000
lstm_ms_l = (lstm_inf_time_long  / n_test) * 1000
bert_ms   = (bert_inf_time       / n_test) * 1000

comparison = pd.DataFrame({
    'Model':               ['BiLSTM (seq=256)', 'BiLSTM (seq=512)', 'DistilBERT (seq=256)'],
    'Test Accuracy':       [f'{lstm_acc_short:.4f}', f'{lstm_acc_long:.4f}', f'{bert_acc:.4f}'],
    'Parameters':          [f'{lstm_params_short:,}', f'{lstm_params_long:,}', f'{bert_params:,}'],
    'Train Time (s)':      [f'{lstm_train_time_short:.1f}', f'{lstm_train_time_long:.1f}', f'{bert_train_time:.1f}'],
    'Inference (ms/item)': [f'{lstm_ms_s:.2f}', f'{lstm_ms_l:.2f}', f'{bert_ms:.2f}'],
    'Parallelizable':      ['No', 'No', 'Yes'],
    'Long-Context Robust': ['Partial', 'No', 'Yes'],
    'Pre-trained':         ['No', 'No', 'Yes'],
})

print('='*110)
print('FINAL COMPARISON: LSTM vs DistilBERT')
print('='*110)
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models     = ['LSTM\n(seq=256)', 'LSTM\n(seq=512)', 'DistilBERT\n(seq=256)']
bar_colors = ['#4C72B0', '#DD8452', '#55A868']

# Accuracy
accs = [lstm_acc_short, lstm_acc_long, bert_acc]
b = axes[0].bar(models, accs, color=bar_colors, edgecolor='white', width=0.5)
axes[0].set_ylim(0.5, 1.0); axes[0].set_title('Test Accuracy', fontweight='bold')
[axes[0].text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.3f}',
              ha='center', fontweight='bold') for bar, v in zip(b, accs)]
axes[0].grid(axis='y', alpha=0.3)

# Training Time
times = [lstm_train_time_short, lstm_train_time_long, bert_train_time]
b2 = axes[1].bar(models, times, color=bar_colors, edgecolor='white', width=0.5)
axes[1].set_title('Training Time (s)', fontweight='bold')
[axes[1].text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.0f}s',
              ha='center', fontweight='bold') for bar, v in zip(b2, times)]
axes[1].grid(axis='y', alpha=0.3)

# Inference Speed
inf_s = [lstm_ms_s, lstm_ms_l, bert_ms]
b3 = axes[2].bar(models, inf_s, color=bar_colors, edgecolor='white', width=0.5)
axes[2].set_title('Inference (ms/sample)', fontweight='bold')
[axes[2].text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}',
              ha='center', fontweight='bold') for bar, v in zip(b3, inf_s)]
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('LSTM vs DistilBERT: Empirical Comparison', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

### Trade-off Analysis

| Dimension | Winner | Reason |
|---|---|---|
| Accuracy | DistilBERT | Pre-trained representations + full bidirectional context |
| Parameters | LSTM | ~20× fewer — no pre-training overhead |
| GPU training speed | DistilBERT | Full sequence parallelism |
| CPU inference speed | LSTM | Sequential ops are computationally lightweight |
| Long-context robustness | DistilBERT | Direct O(1) attention, no vanishing gradient |
| Memory footprint | LSTM | Orders of magnitude smaller |
| Deployment simplicity | LSTM | No tokenizer dependency, runs anywhere |
| State-of-the-art NLP | DistilBERT | Transformers dominate every NLP benchmark |

**When to choose LSTM:**
- Edge / embedded devices (RAM < 512MB)
- Real-time streaming (sequential inference is natural)
- Sequences < 50 tokens (Transformer's O(n²) overhead not justified)
- Training data < 1k samples

**When to choose Transformers:**
- Any sequence > 100 tokens where long-range context matters
- Production NLP with high accuracy requirements
- Fine-tuning available (saves 90% of training cost)
- Multi-task transfer learning scenarios